# 02 - Full pipeline: analysis, features, additive models

Self-contained notebook for **Google Colab**. It reproduces the `src/` pipeline
(`features.py`, `build_dataset.py`, `train.py`, `predict.py`) for the Zindi challenge
*Financial Stress Prediction Challenge (September Edition)*. The full log
of the reasoning, with each experiment and its score, is in `experiments.md`.

**Metric** (formula recovered from two known public scores):
`score = 0.4 × AUC + 0.6 × (1 − logloss / 0.595)`. One thousandth of logloss is therefore
worth one thousandth of score, versus 0.4 thousandth for one thousandth of AUC.

**Approach in brief**
1. Starting point: the Zindi starter notebook (logistic regression, AUC 0.753).
2. Analysis: each client has 7 snapshots, and the target shows up in the break between
   the 3 recent months and the 3 older months.
3. Reverse engineering: the data are stationary and the target is computed from
   the features (a score + noise). This score is a **sum** of a few factors
   (outflow/inflow ratio, drop in inflows, drop in balance, appearance of withdrawals
   or payments, balance volatility, activity). Volumes, max amounts and
   counterparties are noise.
4. The noise added to the score before keeping the 15 % riskiest rows is
   **Gaussian**: P(stress | x) = Φ(score). The models are therefore additive on the
   **probit** scale (a model additive in logit overestimates low risks).
5. Models: three **additive** models on the 15 top-ranked features —
   probit LightGBM with single-split trees (monotonic constraints), probit GAM with
   P-splines (same constraints) and CatBoost with single-split trees — stratified
   cross-validation with 10 folds × 3 seeds, then a blend on the probit scale.

Seed fixed everywhere (`SEED = 42`), as required by the rules.

## 1. Installation and configuration

In [ ]:
!pip install -q lightgbm catboost

In [ ]:
import itertools
import os

import catboost as cb
import lightgbm as lgb
import numpy as np
import pandas as pd
from scipy import optimize, special
from scipy.interpolate import BSpline
from sklearn.metrics import log_loss, roc_auc_score
from sklearn.model_selection import StratifiedKFold

SEED = 42
N_FOLDS = 10
BAGGING_SEEDS = [42, 123, 2024]
# For a quick run, reduce N_FOLDS to 5 and BAGGING_SEEDS to [42]: the resulting score
# will then be slightly different from that of the submission.

## 2. Loading the data

Upload `Train.csv` and `Test.csv` (the challenge's *Data* page on zindi.world). The cell
first looks in `data/raw/`, then offers the Colab upload.

In [ ]:
RAW_DIR = "data/raw" if os.path.exists("data/raw") else "../data/raw"
if os.path.exists(f"{RAW_DIR}/Train.csv") and os.path.exists(f"{RAW_DIR}/Test.csv"):
    train_path, test_path = f"{RAW_DIR}/Train.csv", f"{RAW_DIR}/Test.csv"
else:
    from google.colab import files
    print("Upload Train.csv and Test.csv:")
    files.upload()
    train_path, test_path = "Train.csv", "Test.csv"

train = pd.read_csv(train_path)
test = pd.read_csv(test_path)
TARGET = "liquidity_stress_next_30d"
print(train.shape, test.shape, "stress rate:", train[TARGET].mean())

## 3. Analysis: what guided the feature engineering

Five findings, each verified in `experiments.md`.

**3.1 Client structure.** By grouping rows by profile (ARPU, age,
region, segment…), we find 10,000 clients each having 7 snapshots: 4 in
train, 3 in test. A client's 42 months are all distinct, so there is
**no temporal leakage** between snapshots. The target is nearly independent
from one snapshot to the next, and no feature uses the test data.

In [ ]:
PROFILE = ["arpu", "age", "gender", "region", "smartphone", "segment", "earning_pattern", "x_90_d_activity_rate"]
both = pd.concat([train.assign(_src="train"), test.assign(_src="test")], ignore_index=True)
both["client"] = both.groupby(PROFILE, sort=False).ngroup()
print("clients:", both["client"].nunique())
print("snapshots per client (train + test):", both.groupby("client").size().value_counts().to_dict())
print("stress rate by position in the client block (train):",
      both[both._src == "train"].groupby(both.groupby(["_src", "client"]).cumcount())[TARGET].mean().round(4).to_dict())

**3.2 The target shows up in 3-month blocks.** For stressed snapshots,
the gap with non-stressed ones flips between M4 and M3, and stays flat within
each block. In the recent block, inflows drop and outflows rise.

In [ ]:
y_tr = train[TARGET]
rows = {}
for serie in ["daily_avg_bal", "deposit_total_value", "received_total_value", "withdraw_total_value", "paybill_total_value"]:
    lv = np.log1p(train[[f"m{m}_{serie}" for m in range(6, 0, -1)]])
    rows[serie] = (lv[y_tr == 1].mean() - lv[y_tr == 0].mean()).to_numpy().round(2)
pd.DataFrame(rows, index=["M6", "M5", "M4", "M3", "M2", "M1"]).T

**3.3 Summing flows by family brings out the signal.** The
recent / older inflow ratio (deposits + received + bank transfers) predicts much
better than each type taken alone. When inflows fall below 5 % of their older
level, about 95 % of snapshots are stressed.

In [ ]:
inflow = sum(train[[f"m{m}_{t}_total_value" for m in range(1, 7)]].to_numpy(dtype=float)
             for t in ["deposit", "received", "transfer_from_bank"])
ratio = (inflow[:, :3].mean(1) + 1) / (inflow[:, 3:].mean(1) + 1)
print("Univariate AUC of the inflow ratio:", round(1 - roc_auc_score(y_tr, ratio), 4))
pd.Series(y_tr.to_numpy()).groupby(pd.cut(ratio, [0, 0.05, 0.2, 0.4, 0.6, 0.8, 1.0, 1.5, 1e9])).agg(["mean", "size"]).round(3)

**3.4 The target is computed from the features.** Over all rows,
the recent block and the older block have the same distribution (the generator produces
stationary data), but not among non-stressed rows alone: the generator does not
modify the stressed rows, it labels the rows whose score (a function of the
features + noise) is highest. Only the information used by this score
matters, which guided the search for factors.

In [ ]:
rows = {}
for t in ["deposit", "received", "withdraw"]:
    act = train[[f"m{m}_{t}_total_value" for m in range(1, 7)]].to_numpy() > 0
    rows[t] = {"all rows: recent": act[:, :3].mean(), "all rows: older": act[:, 3:].mean(),
               "non-stressed: recent": act[y_tr == 0, :3].mean(), "non-stressed: older": act[y_tr == 0, 3:].mean()}
pd.DataFrame(rows).T.round(3)

**3.5 Paybill and merchant payments form a single category.** An
appearance of paybill (absent from the 3 older months, present in the 3 recent ones)
changes nothing if merchant payments were already active; payments that
appear when neither of the two existed nearly triple the stress rate.

In [ ]:
def etat(a):
    rec, old = a[:, :3].sum(1), a[:, 3:].sum(1)
    return np.select([(old == 0) & (rec > 0), (old > 0) & (rec == 0), (old == 0) & (rec == 0)],
                     ["started", "stopped", "never"], "active")


pb = train[[f"m{m}_paybill_total_value" for m in range(1, 7)]].to_numpy()
mp = train[[f"m{m}_merchantpay_total_value" for m in range(1, 7)]].to_numpy()
pd.crosstab(etat(pb), etat(mp), values=y_tr, aggfunc="mean").round(3).rename_axis(index="paybill", columns="merchantpay")

## 4. Feature engineering

Code identical to `src/features.py`.

In [ ]:
import numpy as np
import pandas as pd

MONTHS = [1, 2, 3, 4, 5, 6]

# Only metric used per transaction type (see the module docstring).
TRANSACTION_TYPES = {
    "paybill": ["total_value"],
    "merchantpay": ["total_value"],
    "transfer_from_bank": ["total_value"],
    "mm_send": ["total_value"],
    "received": ["total_value"],
    "deposit": ["total_value"],
    "withdraw": ["total_value"],
}

OUTFLOW_TYPES = ["paybill", "merchantpay", "mm_send", "withdraw"]
INFLOW_TYPES = ["deposit", "received", "transfer_from_bank"]

EPS = 1e-6


def _monthly_columns(prefix_fn):
    return [prefix_fn(m) for m in MONTHS]


def _series_features(df: pd.DataFrame, cols: list[str], name: str) -> pd.DataFrame:
    """Level, volatility and momentum (recent block vs older block) of a 6-month series.

    Only the per-3-month-block totals and the 6-month volatility carry signal
    (experiments.md, 2026-10-10): the trend, the last month alone, the M1/M2 ratio
    and the per-block min/max were removed without loss (LightGBM 0.23146 without them
    vs 0.23155 with them).
    """
    values = df[cols].to_numpy(dtype=float)
    recent = values[:, :3]  # M1, M2, M3
    older = values[:, 3:]  # M4, M5, M6

    mean = values.mean(axis=1)
    std = values.std(axis=1)

    out = pd.DataFrame(index=df.index)
    out[f"{name}_sum_6m"] = values.sum(axis=1)
    out[f"{name}_mean_6m"] = mean
    out[f"{name}_std_6m"] = std
    out[f"{name}_cv_6m"] = std / (np.abs(mean) + EPS)
    out[f"{name}_momentum_recent_vs_old"] = (recent.mean(axis=1) + EPS) / (
        older.mean(axis=1) + EPS
    )
    return out


def build_transaction_features(df: pd.DataFrame) -> pd.DataFrame:
    """Level/volatility/momentum features for each (transaction type, metric)."""
    blocks = []
    for ttype, metrics in TRANSACTION_TYPES.items():
        for metric in metrics:
            cols = _monthly_columns(lambda m, t=ttype, me=metric: f"m{m}_{t}_{me}")
            blocks.append(_series_features(df, cols, f"{ttype}_{metric}"))
    return pd.concat(blocks, axis=1)


def build_balance_features(df: pd.DataFrame) -> pd.DataFrame:
    """Level/volatility/momentum features for the average daily balance."""
    cols = _monthly_columns(lambda m: f"m{m}_daily_avg_bal")
    return _series_features(df, cols, "daily_avg_bal")


def _type_total_value_6m(df: pd.DataFrame, ttype: str) -> pd.Series:
    cols = _monthly_columns(lambda m, t=ttype: f"m{m}_{t}_total_value")
    return df[cols].sum(axis=1)


def build_ratio_features(df: pd.DataFrame) -> pd.DataFrame:
    """Cross-transaction-type ratios (outflows/inflows, withdrawals/deposits)."""
    out = pd.DataFrame(index=df.index)

    outflow_value = sum(_type_total_value_6m(df, t) for t in OUTFLOW_TYPES)
    inflow_value = sum(_type_total_value_6m(df, t) for t in INFLOW_TYPES)
    out["outflow_to_inflow_value_ratio"] = (outflow_value + EPS) / (inflow_value + EPS)

    withdraw_value = _type_total_value_6m(df, "withdraw")
    deposit_value = _type_total_value_6m(df, "deposit")
    out["withdraw_to_deposit_value_ratio"] = (withdraw_value + EPS) / (
        deposit_value + EPS
    )
    return out


ALL_TYPES = list(TRANSACTION_TYPES.keys())


def build_overall_activity_features(df: pd.DataFrame) -> pd.DataFrame:
    """Level/volatility/momentum of activity across all types combined: a client can
    switch from one type to another (paybill -> mm_send) without their overall activity
    changing, a signal invisible to per-type features."""
    value_cols = [
        _monthly_columns(lambda m, t=ttype: f"m{m}_{ttype}_total_value")
        for ttype in ALL_TYPES
    ]
    total_value_per_month = sum(df[cols].to_numpy(dtype=float) for cols in value_cols)
    tmp_value = pd.DataFrame(
        total_value_per_month, columns=_monthly_columns(lambda m: f"m{m}"), index=df.index
    )
    return _series_features(tmp_value, tmp_value.columns.tolist(), "overall_value")


def build_share_features(df: pd.DataFrame) -> pd.DataFrame:
    """Share of each type in total 6-month outflows/inflows: captures the
    composition of the behaviour (cash vs bills), not just its level."""
    out = pd.DataFrame(index=df.index)

    outflow_value = sum(_type_total_value_6m(df, t) for t in OUTFLOW_TYPES)
    for t in OUTFLOW_TYPES:
        out[f"{t}_share_of_outflow"] = (_type_total_value_6m(df, t) + EPS) / (
            outflow_value + EPS
        )

    inflow_value = sum(_type_total_value_6m(df, t) for t in INFLOW_TYPES)
    for t in INFLOW_TYPES:
        out[f"{t}_share_of_inflow"] = (_type_total_value_6m(df, t) + EPS) / (
            inflow_value + EPS
        )

    return out


def build_balance_to_income_features(df: pd.DataFrame) -> pd.DataFrame:
    """Balance relative to income (ARPU): a low balance is not equally worrying depending on
    the client's income, which neither variable captures on its own."""
    bal_cols = _monthly_columns(lambda m: f"m{m}_daily_avg_bal")
    bal_mean = df[bal_cols].mean(axis=1)
    out = pd.DataFrame(index=df.index)
    out["balance_to_arpu_ratio"] = bal_mean / (df["arpu"] + EPS)
    return out


FLOW_SMOOTHING = 1.0  # additive smoothing of ratios: prevents a near-zero month from blowing up a ratio


def _block_features(values: np.ndarray, name: str, out: dict) -> None:
    """Compares the recent block (M1-M3) with the older block (M4-M6) of a monthly series.

    The analysis (experiments.md, 2026-10-10) shows that the target shows up at the level of
    3-month blocks: for stressed snapshots, the gap with non-stressed ones
    is nearly constant within each block and flips between M4 and M3.
    """
    rec, old = values[:, :3], values[:, 3:]
    out[f"{name}_rec_mean"] = rec.mean(axis=1)
    out[f"{name}_old_mean"] = old.mean(axis=1)
    out[f"{name}_ratio"] = (rec.mean(axis=1) + FLOW_SMOOTHING) / (old.mean(axis=1) + FLOW_SMOOTHING)
    out[f"{name}_logratio"] = np.log1p(rec).mean(axis=1) - np.log1p(old).mean(axis=1)


def _monthly_array(df: pd.DataFrame, name: str) -> np.ndarray:
    return df[_monthly_columns(lambda m: f"m{m}_{name}")].to_numpy(dtype=float)


def build_flow_aggregate_features(df: pd.DataFrame) -> pd.DataFrame:
    """Aggregates of inflows / outflows and stop / start events.

    A single type taken alone is noisy; summing inflows (deposits + received + transfers) and
    outflows (withdrawals + paybill + merchant payments + sends) brings out the
    signal (univariate AUC of the recent/older ratio: 0.72, vs 0.66 for the best
    single type; inflows below 5% of the older level -> 95% stress).

    The stop / start indicators capture what the ratios smooth out: deposits
    stopped -> 37% stress, withdrawals appearing -> 44%.
    """
    out = {}
    inflow_value = sum(_monthly_array(df, f"{t}_total_value") for t in INFLOW_TYPES)
    outflow_value = sum(_monthly_array(df, f"{t}_total_value") for t in OUTFLOW_TYPES)
    balance = _monthly_array(df, "daily_avg_bal")

    _block_features(inflow_value, "inflow_value", out)
    _block_features(outflow_value, "outflow_value", out)
    _block_features(balance, "bal", out)

    net = inflow_value - outflow_value
    out["net_rec_sum"] = net[:, :3].sum(axis=1)
    out["net_old_sum"] = net[:, 3:].sum(axis=1)
    out["net_rec_over_bal_old"] = net[:, :3].sum(axis=1) / (balance[:, 3:].mean(axis=1) + FLOW_SMOOTHING)
    out["net_rec_over_inflow_old"] = net[:, :3].sum(axis=1) / (inflow_value[:, 3:].sum(axis=1) + FLOW_SMOOTHING)
    out["outflow_over_inflow_rec"] = (outflow_value[:, :3].sum(axis=1) + FLOW_SMOOTHING) / (
        inflow_value[:, :3].sum(axis=1) + FLOW_SMOOTHING
    )
    out["outflow_over_inflow_old"] = (outflow_value[:, 3:].sum(axis=1) + FLOW_SMOOTHING) / (
        inflow_value[:, 3:].sum(axis=1) + FLOW_SMOOTHING
    )

    n_stopped = np.zeros(len(df))
    n_started = np.zeros(len(df))
    for t in TRANSACTION_TYPES:
        value = _monthly_array(df, f"{t}_total_value")
        rec_zero = (value[:, :3] == 0).all(axis=1)
        old_zero = (value[:, 3:] == 0).all(axis=1)
        stopped = (rec_zero & ~old_zero).astype(float)
        started = (~rec_zero & old_zero).astype(float)
        out[f"{t}_stopped"] = stopped
        out[f"{t}_started"] = started
        out[f"{t}_rec_zero_months"] = (value[:, :3] == 0).sum(axis=1)
        out[f"{t}_old_zero_months"] = (value[:, 3:] == 0).sum(axis=1)
        n_stopped += stopped
        n_started += started
    out["n_types_stopped"] = n_stopped
    out["n_types_started"] = n_started
    out["inflow_types_stopped"] = sum(out[f"{t}_stopped"] for t in INFLOW_TYPES)
    out["outflow_types_started"] = sum(out[f"{t}_started"] for t in OUTFLOW_TYPES)
    return pd.DataFrame(out, index=df.index)


# Paybill and merchant payments behave like a single "payments" category:
# an appearance of paybill when merchant payments are already active leaves
# stress at the base rate (14%), vs 34% when both appear from nothing
# (40% if together). Adding withdrawals or sends to this group brings nothing (tested).
CATEGORY_GROUPS = {
    "payments": ["paybill", "merchantpay"],
}


def build_category_features(df: pd.DataFrame) -> pd.DataFrame:
    """Recent block / older block comparisons per grouped category, and smoothed ratios.

    - Per category (CATEGORY_GROUPS): block comparisons (_block_features),
      start/stop, active months per block.
    - Per type: ratio of block means smoothed at 1 monetary unit (FLOW_SMOOTHING),
      vs 1e-6 for the `_momentum_recent_vs_old` ratios — with a smoothing of 1, an
      appearance is worth its recent amount, comparable to an increase of the same size
      (AUC 0.82 for the amount of withdrawals that appear).
    - Payments and withdrawals relative to the inflows of each block, on the same principle
      as `outflow_over_inflow_rec`, the model's most important feature.
    - Simultaneous stop of deposits and receipts (79% stress, vs 37% and 28% for
      each taken alone).
    """
    out = {}
    for name, types in CATEGORY_GROUPS.items():
        values = sum(_monthly_array(df, f"{t}_total_value") for t in types)
        _block_features(values, f"{name}_value", out)
        rec_sum, old_sum = values[:, :3].sum(axis=1), values[:, 3:].sum(axis=1)
        out[f"{name}_started"] = ((old_sum == 0) & (rec_sum > 0)).astype(float)
        out[f"{name}_stopped"] = ((rec_sum == 0) & (old_sum > 0)).astype(float)
        out[f"{name}_rec_active_months"] = (values[:, :3] > 0).sum(axis=1)
        out[f"{name}_old_active_months"] = (values[:, 3:] > 0).sum(axis=1)

    for t in TRANSACTION_TYPES:
        values = _monthly_array(df, f"{t}_total_value")
        out[f"{t}_block_ratio"] = (values[:, :3].mean(axis=1) + FLOW_SMOOTHING) / (
            values[:, 3:].mean(axis=1) + FLOW_SMOOTHING
        )

    inflow = sum(_monthly_array(df, f"{t}_total_value") for t in INFLOW_TYPES)
    payments = sum(_monthly_array(df, f"{t}_total_value") for t in CATEGORY_GROUPS["payments"])
    withdraw = _monthly_array(df, "withdraw_total_value")
    for block, cols in [("rec", slice(0, 3)), ("old", slice(3, 6))]:
        inflow_sum = inflow[:, cols].sum(axis=1) + FLOW_SMOOTHING
        out[f"payments_over_inflow_{block}"] = (payments[:, cols].sum(axis=1) + FLOW_SMOOTHING) / inflow_sum
        out[f"withdraw_over_inflow_{block}"] = (withdraw[:, cols].sum(axis=1) + FLOW_SMOOTHING) / inflow_sum

    dep_rec = _monthly_array(df, "deposit_total_value") + _monthly_array(df, "received_total_value")
    out["deposit_received_stopped"] = (
        (dep_rec[:, :3].sum(axis=1) == 0) & (dep_rec[:, 3:].sum(axis=1) > 0)
    ).astype(float)
    return pd.DataFrame(out, index=df.index)


PASSTHROUGH_COLUMNS = [
    "arpu",
    "age",
    "x_90_d_activity_rate",
    "gender",
    "region",
    "smartphone",
    "segment",
    "earning_pattern",
]


GROUP_RELATIVE_COLS = [
    "daily_avg_bal_mean_6m",
    "outflow_to_inflow_value_ratio",
    "balance_to_arpu_ratio",
]


def fit_group_stats(features_df: pd.DataFrame, group_col: str = "segment", cols=None):
    """Computes mean/std per group (segment) on the train set ONLY,
    like a scaler: fit on train, then applied as is to test (apply_group_relative_features)
    so that a train feature never depends on the test distribution."""
    cols = cols or GROUP_RELATIVE_COLS
    return features_df.groupby(group_col)[cols].agg(["mean", "std"])


def apply_group_relative_features(
    features_df: pd.DataFrame, group_stats, group_col: str = "segment", cols=None
) -> pd.DataFrame:
    """Z-score of a few key features relative to their group (segment): a
    low balance or ratio does not mean the same thing for an HVC client as for an LVC one."""
    cols = cols or GROUP_RELATIVE_COLS
    out = pd.DataFrame(index=features_df.index)
    for col in cols:
        means = features_df[group_col].map(group_stats[(col, "mean")])
        stds = features_df[group_col].map(group_stats[(col, "std")])
        z = (features_df[col] - means) / (stds.replace(0, np.nan) + EPS)
        out[f"{col}_z_by_segment"] = z.fillna(0.0)
    return out


def build_segment_cross_feature(df: pd.DataFrame) -> pd.DataFrame:
    """Segment x earning_pattern cross: trees only recover an
    interaction between two categoricals if the depth allows it. We
    provide it directly so as not to depend on the chosen depth."""
    out = pd.DataFrame(index=df.index)
    out["segment_x_earning_pattern"] = (
        df["segment"].astype(str) + "_" + df["earning_pattern"].astype(str)
    )
    return out


def build_features(df: pd.DataFrame) -> pd.DataFrame:
    """Builds the per-client feature dataset from the raw Zindi data."""
    blocks = [
        df[PASSTHROUGH_COLUMNS],
        build_transaction_features(df),
        build_balance_features(df),
        build_ratio_features(df),
        build_overall_activity_features(df),
        build_share_features(df),
        build_balance_to_income_features(df),
        build_segment_cross_feature(df),
        build_flow_aggregate_features(df),
        build_category_features(df),
    ]
    features = pd.concat(blocks, axis=1)
    features.index = df["ID"] if "ID" in df.columns else df.index
    return features

In [ ]:
feats_train = build_features(train)
feats_test = build_features(test)

# Z-scores by segment: statistics fitted on the train set only (like a
# scaler), then applied as is to test.
group_stats = fit_group_stats(feats_train)
feats_train = pd.concat([feats_train, apply_group_relative_features(feats_train, group_stats)], axis=1)
feats_test = pd.concat([feats_test, apply_group_relative_features(feats_test, group_stats)], axis=1)

y = train[TARGET].to_numpy()
cat_cols = [c for c in feats_train.columns if feats_train[c].dtype == object or str(feats_train[c].dtype) in ("str", "string")]
for c in cat_cols:
    feats_train[c] = feats_train[c].astype("category")
    feats_test[c] = feats_test[c].astype(pd.CategoricalDtype(categories=feats_train[c].cat.categories))

assert feats_train.isna().sum().sum() == 0 and feats_test.isna().sum().sum() == 0
print(f"{feats_train.shape[1]} features, {len(feats_train)} train rows, {len(feats_test)} test rows")

## 5. Models

Stratified cross-validation: there is no date column and no chronology
between rows, train/test adversarial validation gives an AUC of 0.50 and
the column-by-column leakage check finds nothing (max univariate AUC 0.57).
The hyperparameters are those of `src/train.py` (`FINAL_*_PARAMS`).

**Additive models.** With these features, the hidden score is a sum of functions
of a single feature: single-split trees (`max_depth=1` / `depth=1`)
do better than deeper trees, which only add variance (LightGBM,
5 folds: depth 1 0.2310, 2 0.2316, 3 0.2324). Enforcing the direction of variation of
the factors whose shape function is monotonic further reduces the noise.

**Probit link.** The target is 1 for the 15 % of rows whose "hidden score + noise"
is highest (exactly 1,500 out of 10,000 at each position of the client block). If
this noise is Gaussian, P(y=1|x) = Φ(s(x) − threshold): the score is additive on the probit
scale, not on the logit scale. Trained with the logit objective, the additive LightGBM
overestimates low risks (0.72 % predicted vs 0.39 % observed, a 3 σ gap);
trained with the probit likelihood (custom objective below), it is
calibrated in every risk bucket and gains 0.0007 of logloss (5 folds).

**Probit GAM.** Second model: the same single-feature functions, estimated with
smooth curves (P-splines) rather than step functions, with the same monotonicity
constraints. Code identical to `src/gam.py`.

In [ ]:
import numpy as np
from scipy import optimize, special
from scipy.interpolate import BSpline
from threadpoolctl import threadpool_limits

SPLINE_DEGREE = 3


def _rank_transform(train_sorted, x):
    """Average rank of x among the train values (ties included), scaled to [0, 1]."""
    below = np.searchsorted(train_sorted, x, side="left")
    below_or_equal = np.searchsorted(train_sorted, x, side="right")
    return (below + below_or_equal) / (2 * len(train_sorted))


class ProbitGAM:
    """Additive probit model with P-splines, with optional monotonicity constraints.

    `monotone`: list aligned with the columns, 1 (increasing), -1 (decreasing) or
    0 (unconstrained).
    """

    def __init__(self, n_knots=20, lam=5.0, monotone=None, max_iter=5000):
        self.n_knots = n_knots
        self.lam = lam
        self.monotone = monotone
        self.max_iter = max_iter

    def _raw_basis(self, j, x):
        u = np.clip(_rank_transform(self.train_sorted_[j], x), 0.0, 1.0 - 1e-12)
        knots = np.r_[[0.0] * SPLINE_DEGREE, np.linspace(0, 1, self.n_knots + 2), [1.0] * SPLINE_DEGREE]
        return BSpline.design_matrix(u, knots, SPLINE_DEGREE).toarray()

    def _basis(self, j, x):
        return self._raw_basis(j, x) - self.basis_means_[j]

    def fit(self, X, y):
        with threadpool_limits(limits=1, user_api="blas"):
            return self._fit(X, y)

    def _fit(self, X, y):
        X, y = np.asarray(X, dtype=float), np.asarray(y, dtype=float)
        n, p = X.shape
        n_basis = self.n_knots + SPLINE_DEGREE + 1
        self.train_sorted_ = [np.sort(X[:, j]) for j in range(p)]
        self.basis_means_ = [self._raw_basis(j, X[:, j]).mean(axis=0) for j in range(p)]

        # Coefficients of feature j: beta_j = [0, cumsum(delta_j)]. The curve is
        # monotonic if all increments delta_j have the same sign (L-BFGS-B bounds).
        # Since the basis is centered, the first coefficient is fixed at 0 without loss.
        cumsum = np.vstack([np.zeros(n_basis - 1), np.tril(np.ones((n_basis - 1, n_basis - 1)))])
        second_diff = np.diff(np.eye(n_basis), 2, axis=0) @ cumsum
        penalty = second_diff.T @ second_diff
        design = np.hstack([self._basis(j, X[:, j]) @ cumsum for j in range(p)])
        k = n_basis - 1

        bounds = [(None, None)]
        for j in range(p):
            direction = 0 if self.monotone is None else self.monotone[j]
            bounds += [(0, None) if direction > 0 else (None, 0) if direction < 0 else (None, None)] * k

        def objective(theta):
            eta = theta[0] + design @ theta[1:]
            log_cdf, log_sf = special.log_ndtr(eta), special.log_ndtr(-eta)
            log_pdf = -0.5 * eta**2 - 0.5 * np.log(2 * np.pi)
            nll = -np.sum(np.where(y == 1, log_cdf, log_sf))
            grad_eta = np.where(y == 1, -np.exp(log_pdf - log_cdf), np.exp(log_pdf - log_sf))
            deltas = theta[1:].reshape(p, k)
            pen_grad = self.lam * deltas @ penalty
            value = nll + 0.5 * np.sum(pen_grad * deltas)
            grad = np.r_[grad_eta.sum(), design.T @ grad_eta + pen_grad.ravel()]
            return value / n, grad / n

        theta0 = np.r_[special.ndtri(y.mean()), np.zeros(p * k)]
        result = optimize.minimize(objective, theta0, jac=True, method="L-BFGS-B", bounds=bounds,
                                   options={"maxiter": self.max_iter, "maxfun": 2 * self.max_iter})
        self.intercept_ = result.x[0]
        self.coef_ = [cumsum @ d for d in result.x[1:].reshape(p, k)]
        self.n_iter_ = result.nit
        return self

    def decision_function(self, X):
        """Score η on the probit scale."""
        X = np.asarray(X, dtype=float)
        eta = np.full(len(X), self.intercept_)
        with threadpool_limits(limits=1, user_api="blas"):
            for j, beta in enumerate(self.coef_):
                eta += self._basis(j, X[:, j]) @ beta
        return eta

    def predict_proba(self, X):
        p = special.ndtr(self.decision_function(X))
        return np.column_stack([1 - p, p])

In [ ]:
LGBM_PARAMS = dict(objective="binary", n_estimators=20000, learning_rate=0.05, max_depth=1, num_leaves=2,
                   min_child_samples=64, subsample=0.89, subsample_freq=1, colsample_bytree=0.5,
                   reg_lambda=0.013, verbosity=-1)
LGBM_ESR = 500
CATBOOST_PARAMS = dict(loss_function="Logloss", eval_metric="Logloss", iterations=20000, learning_rate=0.08,
                       depth=1, l2_leaf_reg=11.73, random_strength=0.0625, bagging_temperature=2.42, verbose=False)
CATBOOST_ESR = 500
GAM_KNOTS, GAM_LAMBDA = 20, 5.0
# Enforced directions of variation (1: risk increases with the feature, -1: it decreases): financial
# logic, confirmed by the shape functions of the unconstrained additive model.
MONOTONE = {"outflow_over_inflow_rec": 1, "withdraw_over_inflow_rec": 1, "payments_value_ratio": 1,
            "daily_avg_bal_cv_6m": 1, "inflow_value_ratio": -1, "daily_avg_bal_momentum_recent_vs_old": -1,
            "bal_ratio": -1, "net_rec_over_inflow_old": -1, "x_90_d_activity_rate": -1}
LGBM_MAX_BIN = 1023


def zindi_score(ll, auc):
    return 0.4 * auc + 0.6 * (1 - ll / 0.595)


def folds_for(seed):
    return list(StratifiedKFold(N_FOLDS, shuffle=True, random_state=seed).split(feats_train, y))


def probit_grad_hess(eta, y_):
    # Gradient and Hessian, w.r.t. eta, of the logloss -[y log Φ(eta) + (1-y) log(1-Φ(eta))], computed in log space.
    log_cdf, log_sf = special.log_ndtr(eta), special.log_ndtr(-eta)
    log_pdf = -0.5 * eta**2 - 0.5 * np.log(2 * np.pi)
    r1, r0 = np.exp(log_pdf - log_cdf), np.exp(log_pdf - log_sf)
    grad = np.where(y_ == 1, -r1, r0)
    hess = np.where(y_ == 1, r1 * (r1 + eta), r0 * (r0 - eta))
    return grad, hess


def probit_logloss(y_, eta):
    return -np.mean(np.where(y_ == 1, special.log_ndtr(eta), special.log_ndtr(-eta)))


def lgb_probit_objective(preds, data):
    # preds: current LightGBM score, which already includes the Dataset's init_score.
    return probit_grad_hess(preds, data.get_label())


def lgb_probit_metric(preds, data):
    return "probit_logloss", probit_logloss(data.get_label(), preds), False


def run_lgbm(seed):
    Xtr, Xte = feats_train[selected], feats_test[selected]
    params = {k: v for k, v in LGBM_PARAMS.items() if k not in ("objective", "n_estimators")}
    params.update(max_bin=LGBM_MAX_BIN, monotone_constraints=[MONOTONE.get(c, 0) for c in selected],
                  random_state=seed, objective=lgb_probit_objective, metric="None")
    oof, test_pred = np.zeros(len(y)), np.zeros(len(feats_test))
    for tri, vai in folds_for(seed):
        init = special.ndtri(y[tri].mean())  # the score starts from Φ⁻¹(positive rate)
        dtr = lgb.Dataset(Xtr.iloc[tri], y[tri].astype(float), init_score=np.full(len(tri), init),
                          categorical_feature=selected_cats, params={"max_bin": LGBM_MAX_BIN})
        dva = lgb.Dataset(Xtr.iloc[vai], y[vai].astype(float), init_score=np.full(len(vai), init), reference=dtr)
        bst = lgb.train(params, dtr, num_boost_round=LGBM_PARAMS["n_estimators"], valid_sets=[dva],
                        feval=lgb_probit_metric, callbacks=[lgb.early_stopping(LGBM_ESR, verbose=False)])
        oof[vai] = special.ndtr(bst.predict(Xtr.iloc[vai], num_iteration=bst.best_iteration) + init)
        test_pred += special.ndtr(bst.predict(Xte, num_iteration=bst.best_iteration) + init) / N_FOLDS
    return oof, test_pred


def run_gam(seed):
    Xtr, Xte = feats_train[selected], feats_test[selected]
    mono = [MONOTONE.get(c, 0) for c in selected]
    oof, test_pred = np.zeros(len(y)), np.zeros(len(feats_test))
    for tri, vai in folds_for(seed):
        m = ProbitGAM(n_knots=GAM_KNOTS, lam=GAM_LAMBDA, monotone=mono).fit(Xtr.iloc[tri], y[tri])
        oof[vai] = m.predict_proba(Xtr.iloc[vai])[:, 1]
        test_pred += m.predict_proba(Xte)[:, 1] / N_FOLDS
    return oof, test_pred


def rank_features():
    # Cumulative gain of the additive LightGBM (logit objective) over 5 folds (fixed seed): same ranking as
    # src/train.py. The same ranking done with the probit objective keeps the same 15 features.
    gain = np.zeros(feats_train.shape[1])
    for tri, vai in StratifiedKFold(5, shuffle=True, random_state=SEED).split(feats_train, y):
        m = lgb.LGBMClassifier(**LGBM_PARAMS, random_state=SEED)
        m.fit(feats_train.iloc[tri], y[tri], eval_set=[(feats_train.iloc[vai], y[vai])],
              categorical_feature=cat_cols, callbacks=[lgb.early_stopping(LGBM_ESR, verbose=False)])
        gain += m.booster_.feature_importance("gain")
    return pd.Series(gain, index=feats_train.columns).sort_values(ascending=False)


def run_catboost(seed):
    Xtr, Xte = feats_train[selected], feats_test[selected]
    oof, test_pred = np.zeros(len(y)), np.zeros(len(feats_test))
    cat_idx = [Xtr.columns.get_loc(c) for c in selected_cats]
    for tri, vai in folds_for(seed):
        m = cb.CatBoostClassifier(**CATBOOST_PARAMS, random_seed=seed)
        m.fit(Xtr.iloc[tri], y[tri], eval_set=(Xtr.iloc[vai], y[vai]), cat_features=cat_idx,
              early_stopping_rounds=CATBOOST_ESR, verbose=False)
        oof[vai] = m.predict_proba(Xtr.iloc[vai])[:, 1]
        test_pred += m.predict_proba(Xte)[:, 1] / N_FOLDS
    return oof, test_pred


def bagged(run_fn, name):
    runs = [run_fn(s) for s in BAGGING_SEEDS]
    oof = np.mean([r[0] for r in runs], axis=0)
    test_pred = np.mean([r[1] for r in runs], axis=0)
    ll, auc = log_loss(y, oof), roc_auc_score(y, oof)
    print(f"{name:16s} logloss={ll:.5f} auc={auc:.5f} estimated score={zindi_score(ll, auc):.5f}")
    return oof, test_pred

**Feature selection.** The 15 top-ranked features carry all the
signal: beyond that, we only add variance (selection redone within each fold, without
bias: 0.2298 with 15 features vs 0.2305 with all 137, see experiments.md).

In [ ]:
N_FEATURES = 15
ranking = rank_features()
selected = list(ranking.index[:N_FEATURES])
selected_cats = [c for c in cat_cols if c in selected]
print(f"{len(selected)} selected features:", ", ".join(selected))

In [ ]:
oof_lgbm, test_lgbm = bagged(run_lgbm, "LightGBM probit")
oof_gam, test_gam = bagged(run_gam, "GAM probit")
oof_cb, test_cb = bagged(run_catboost, "CatBoost")

## 6. Blend on the probit scale

The three models are averaged on the probit scale (weighted average of Φ⁻¹(p)):
a weighted sum of additive models remains additive. The weights (grid with a step of
0.05, summing to 1) maximize the Zindi score on the OOF predictions. No
recalibration: the blend is calibrated in every risk bucket (table
below), and a Platt recalibration fitted in cross-validation gains nothing.

In [ ]:
oof_all = {"lightgbm": oof_lgbm, "gam": oof_gam, "catboost": oof_cb}
test_all = {"lightgbm": test_lgbm, "gam": test_gam, "catboost": test_cb}


def probit_blend(preds, weights):
    eta = sum(w * special.ndtri(np.clip(preds[name], 1e-15, 1 - 1e-15)) for name, w in weights.items())
    return special.ndtr(eta)


grid = np.round(np.arange(0.0, 1.0 + 1e-9, 0.05), 10)
best_score, weights = -np.inf, None
for w_lgbm, w_gam in itertools.product(grid, repeat=2):
    if w_lgbm + w_gam > 1.0 + 1e-9:
        continue
    candidate = {"lightgbm": w_lgbm, "gam": w_gam, "catboost": max(0.0, round(1.0 - w_lgbm - w_gam, 10))}
    p = probit_blend(oof_all, candidate)
    score = zindi_score(log_loss(y, p), roc_auc_score(y, p))
    if score > best_score:
        best_score, weights = score, candidate

oof_blend = probit_blend(oof_all, weights)
test_pred = probit_blend(test_all, weights)
ll, auc = log_loss(y, oof_blend), roc_auc_score(y, oof_blend)
print("weights:", {k: float(v) for k, v in weights.items()})
print(f"blend probit logloss={ll:.5f} auc={auc:.5f} estimated score={zindi_score(ll, auc):.5f}")

calib = pd.DataFrame({"y": y, "p": oof_blend})
calib = calib.groupby(pd.cut(calib.p, [0, .005, .01, .02, .05, .1, .2, .4, .6, .8, .9, .95, .99, 1]), observed=True).agg(
    n=("y", "size"), predit=("p", "mean"), observe=("y", "mean"))
calib["ecart_en_sigma"] = (calib.observe - calib.predit) / np.sqrt(calib.predit * (1 - calib.predit) / calib.n)
calib.round(4)

## 7. Submission

Columns `ID, Target`, with raw probabilities, never rounded or thresholded
(challenge rule).

In [ ]:
submission = pd.DataFrame({"ID": feats_test.index, "Target": test_pred})
assert submission.shape == (len(test), 2) and submission["Target"].between(0, 1).all()
submission.to_csv("submission.csv", index=False)
print(submission.describe())
try:
    from google.colab import files
    files.download("submission.csv")
except ImportError:
    print("File saved: submission.csv")